# Phase 2 (production) — Fine-tune DistilBERT on a free Colab T4

This notebook runs the real `src/training/train_transformer.py` script against a pretrained
DistilBERT (or FinBERT) checkpoint from the HuggingFace Hub. It exists because the sandbox
this repo was originally scaffolded in has no network path to `huggingface.co` — see the
README's "Sandbox execution note". Run this notebook (Runtime → Change runtime type → T4 GPU)
to produce the real fine-tuned numbers this project's README quotes as the production result.

**Steps:** clone the repo → install deps → run each variant → MLflow runs land in `./mlruns`
→ zip and download `mlruns/` → merge into your local repo's `mlruns/` so Phase 3's registry
step can see every run (sandbox-demo and production) in one place.

In [ ]:
!git clone https://github.com/<your-username>/complaint-intelligence-platform.git
%cd complaint-intelligence-platform
!pip install -q -r requirements.txt

In [ ]:
# If you haven't already: acquire + preprocess real CFPB data (Colab has full internet access)
!python -m src.data.acquire --n 120000 --out data/raw/complaints.csv
!python -m src.data.preprocess --raw data/raw/complaints.csv --out-dir data/processed

In [ ]:
# 4-6 real variants: base model, learning rate, weighting strategy, max sequence length.
# Each call is one MLflow run under the same experiment name the sandbox demo used, so
# Phase 3's promotion gate can compare all of them side by side.
!python -m src.training.train_transformer --base-model distilbert-base-uncased --weighting inverse_freq --lr 2e-5 --epochs 3 --run-name hf_distilbert_invfreq_lr2e-5
!python -m src.training.train_transformer --base-model distilbert-base-uncased --weighting inverse_freq --lr 5e-5 --epochs 3 --run-name hf_distilbert_invfreq_lr5e-5
!python -m src.training.train_transformer --base-model distilbert-base-uncased --weighting none         --lr 2e-5 --epochs 3 --run-name hf_distilbert_unweighted
!python -m src.training.train_transformer --base-model distilbert-base-uncased --weighting inverse_freq --lr 2e-5 --epochs 3 --max-length 128 --run-name hf_distilbert_shortseq
!python -m src.training.train_transformer --base-model ProsusAI/finbert       --weighting inverse_freq --lr 2e-5 --epochs 3 --run-name hf_finbert_invfreq_lr2e-5

In [ ]:
# Inspect results, then zip mlruns/ + models/candidates/ and download for Phase 3
!python -c "import mlflow; [print(r) for r in mlflow.search_runs(experiment_names=['complaint-classification-finetune']).to_dict('records')]"
!zip -r mlflow_and_models.zip mlruns models/candidates
from google.colab import files
files.download('mlflow_and_models.zip')